In [34]:
import pandas as pd

### 1.Carga de Datos

In [35]:
df = pd.read_csv("../data/raw/2019-Nov.csv", nrows=1000000)

df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


### 2. Limpieza de datos

#### Validación de precios
Se eliminaron registros con precio igual a 0 para evitar distorsiones en el cálculo de revenue.

In [36]:
df["price"].describe()

count    1000000.000000
mean         292.181443
std          347.585961
min            0.000000
25%           69.720000
50%          172.180000
75%          361.970000
max         2574.070000
Name: price, dtype: float64

In [37]:
df[df["price"] == 0].head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
6258,2019-11-01 00:38:01 UTC,view,33100000,2058719826188173878,NaN,NaN,0.0,546996930,969ea68f-a919-4d32-8925-b709d87c539c
7245,2019-11-01 00:42:51 UTC,view,33100000,2058719826188173878,NaN,NaN,0.0,546996930,b1ab3863-bbf5-4370-a3d8-bfa70580dd09
12743,2019-11-01 01:07:15 UTC,view,12720812,2053013553559896355,NaN,NaN,0.0,516269492,9bf68f2a-fd78-4b19-97ac-7d838c82eb4c
12908,2019-11-01 01:07:58 UTC,view,12720812,2053013553559896355,NaN,NaN,0.0,516269492,9bf68f2a-fd78-4b19-97ac-7d838c82eb4c
13503,2019-11-01 01:26:19 UTC,view,38900075,2085718636156158307,NaN,NaN,0.0,539587206,4d91487b-2b8d-41e3-9328-f8ab325a95cf


In [38]:
df = df[df["price"] > 0]

#### Validación de usuarios
Se eliminaron registros sin user_id para permitir análisis de comportamiento y churn a nivel usuario.

In [39]:
df["user_id"].isna().sum()

np.int64(0)

In [ ]:
df = df.dropna(subset=["user_id"])  # Asegura usuarios válidos

### 3.Guardado intermedio (muestra)

In [41]:
df.to_csv("../data/processed/events_sample.csv", index=False)

### 4.Transformaciones

In [42]:
df_work = pd.read_csv("../data/processed/events_sample.csv")

In [43]:
df_work.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [44]:
df_work.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999027 entries, 0 to 999026
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     999027 non-null  object 
 1   event_type     999027 non-null  object 
 2   product_id     999027 non-null  int64  
 3   category_id    999027 non-null  int64  
 4   category_code  682534 non-null  object 
 5   brand          852803 non-null  object 
 6   price          999027 non-null  float64
 7   user_id        999027 non-null  int64  
 8   user_session   999027 non-null  object 
dtypes: float64(1), int64(3), object(5)
memory usage: 68.6+ MB


### Formatear las fechas a datetime

In [46]:
df_work["event_time"] = pd.to_datetime(df_work["event_time"])

#### NaN categóricos

In [47]:
df_work["category_code"] = df_work["category_code"].fillna("unknown")
df_work["brand"] = df_work["brand"].fillna("unknown")

### 5.Validación final

In [49]:
df_work[["category_code", "brand"]].isna().sum()

category_code    0
brand            0
dtype: int64

In [50]:
df_work.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999027 entries, 0 to 999026
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype              
---  ------         --------------   -----              
 0   event_time     999027 non-null  datetime64[ns, UTC]
 1   event_type     999027 non-null  object             
 2   product_id     999027 non-null  int64              
 3   category_id    999027 non-null  int64              
 4   category_code  999027 non-null  object             
 5   brand          999027 non-null  object             
 6   price          999027 non-null  float64            
 7   user_id        999027 non-null  int64              
 8   user_session   999027 non-null  object             
dtypes: datetime64[ns, UTC](1), float64(1), int64(3), object(4)
memory usage: 68.6+ MB


### 6.Guardado final

In [51]:
df_work.to_csv("../data/processed/events_clean.csv", index=False)
df_work.to_parquet("../data/processed/events_clean.parquet", index=False)

### 7.Dataset final

In [57]:
df_final = pd.read_parquet("../data/processed/events_clean.parquet")
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999027 entries, 0 to 999026
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype              
---  ------         --------------   -----              
 0   event_time     999027 non-null  datetime64[ns, UTC]
 1   event_type     999027 non-null  object             
 2   product_id     999027 non-null  int64              
 3   category_id    999027 non-null  int64              
 4   category_code  999027 non-null  object             
 5   brand          999027 non-null  object             
 6   price          999027 non-null  float64            
 7   user_id        999027 non-null  int64              
 8   user_session   999027 non-null  object             
dtypes: datetime64[ns, UTC](1), float64(1), int64(3), object(4)
memory usage: 68.6+ MB


In [58]:
df_final.describe()

,product_id,category_id,price,user_id
count,9.990270e+05,9.990270e+05,999027.000000,9.990270e+05
mean,1.059918e+07,2.057536e+18,292.466013,5.352248e+08
std,1.199383e+07,1.889096e+16,347.635501,2.014576e+07
min,1.000978e+06,2.053014e+18,0.770000,2.749691e+08
25%,1.005186e+06,2.053014e+18,70.010000,5.159505e+08
50%,4.900181e+06,2.053014e+18,172.200000,5.307941e+08
75%,1.570018e+07,2.053014e+18,361.970000,5.544903e+08
max,6.060000e+07,2.180737e+18,2574.070000,5.665059e+08
